In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os, json

from matminer.featurizers.composition import ElementFraction, ElementProperty
from matminer.featurizers.conversions import StrToComposition

In [2]:
DATA_CSV = "./data/mxene_real.csv"
df = pd.read_csv(DATA_CSV)
df.head()

,mxene,gibbs_free_energy
0,CrCrBF2,2.381070
1,CrHfCF2,2.280670
2,CrMoCF2,1.239013
3,CrMoNCl2,2.416119
4,CrMoNO2,-0.640507


In [3]:
# (1) Convert string formula → pymatgen Composition object
stc = StrToComposition(target_col_id="composition_obj")
df_feat = stc.featurize_dataframe(df, "mxene", ignore_errors=True)

# (2) Compute elemental fractions
ef = ElementFraction()
df_feat = ef.featurize_dataframe(df_feat, "composition_obj", ignore_errors=True)

# (3) Magpie elemental statistics
magpie = ElementProperty.from_preset(preset_name="magpie")
df_feat = magpie.featurize_dataframe(df_feat, "composition_obj", ignore_errors=True)

# Collect feature columns
non_feature_cols = {
    "mxene","gibbs_free_energy","composition_obj"
}
feature_cols = [c for c in df_feat.columns if c not in non_feature_cols]
print("n_features:", len(feature_cols))

StrToComposition:   0%|          | 0/538 [00:00<?, ?it/s]

ElementFraction:   0%|          | 0/538 [00:00<?, ?it/s]

/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/538 [00:00<?, ?it/s]

n_features: 250


In [4]:
# See and save
df_feat.to_csv("./data/train_real.csv", index=False)
print("Saved featurized data to train_real.csv")
df_feat.head()


Saved featurized data to train_real.csv


,mxene,gibbs_free_energy,composition_obj,H,He,Li,Be,B,C,N,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,CrCrBF2,2.381070,"(Cr, B, F)",0.0,0.0,0.0,0.0,0.2,0.0,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,130.8,92.64,15.0
1,CrHfCF2,2.280670,"(Cr, Hf, C, F)",0.0,0.0,0.0,0.0,0.0,0.2,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,129.4,91.52,15.0
2,CrMoCF2,1.239013,"(Cr, Mo, C, F)",0.0,0.0,0.0,0.0,0.0,0.2,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,136.4,97.12,15.0
3,CrMoNCl2,2.416119,"(Cr, Mo, N, Cl)",0.0,0.0,0.0,0.0,0.0,0.0,0.2,...,0.0,0.0,0.0,0.0,64.0,229.0,165.0,156.0,73.60,64.0
4,CrMoNO2,-0.640507,"(Cr, Mo, N, O)",0.0,0.0,0.0,0.0,0.0,0.0,0.2,...,0.0,0.0,0.0,0.0,12.0,229.0,217.0,135.2,98.56,12.0
